XGBoost (eXtreme Gradient Boosting) is an optimized distributed gradient boosting library based on decision trees that computes a second-order Taylor expansion of the loss function. For classification tasks, it sequentially adds weak decision tree learners to minimize objective loss while controlling model complexity through explicit L1/L2 regularization.

---

## Core Architecture & Classification Objectives

XGBoost constructs trees level-wise (or histogram-wise using `hist`) by iteratively minimizing a regularized objective function:

$$\mathcal{L}^{(t)} = \sum_{i=1}^n l(y_i, \hat{y}_i^{(t-1)} + f_t(x_i)) + \Omega(f_t)$$

where the regularization term $\Omega(f) = \gamma T + \frac{1}{2}\lambda \sum_{j=1}^T w_j^2 + \alpha \sum_{j=1}^T \vert{}w_j\vert{}$ constrains tree depth, number of leaves $T$, and leaf weights $w$.

### Classification Objectives (`objective`)

* **`binary:logistic`**: Standard binary classification returning predicted probabilities.
* **`binary:logitraw`**: Binary classification returning raw margin scores prior to logistic transformation.
* **`multi:softprob`**: Multi-class classification returning predicted probability distributions across all classes (requires setting `num_class`).
* **`multi:softmax`**: Multi-class classification returning discrete predicted class labels (requires setting `num_class`).

---

## Comprehensive Hyperparameter Reference

XGBoost parameters are divided into General, Booster, and Task parameters.

### Master Parameter Table

| Parameter | Default | Typical Range | Impact / Tuning Note |
| --- | --- | --- | --- |
| `n_estimators` | `100` | `50–1000+` | Total boosting rounds (number of trees). Increase when lowering `learning_rate`. |
| `learning_rate` (`eta`) | `0.3` | `0.01–0.2` | Step size shrinkage to prevent overfitting. Smaller values require higher `n_estimators`. |
| `max_depth` | `6` | `3–10` | Maximum tree depth. Lower values control overfitting; higher values capture interactions. |
| `min_child_weight` | `1` | `1–10+` | Minimum sum of instance weight (Hessian) needed in a child leaf. Higher values restrict over-splitting. |
| `gamma` (`min_split_loss`) | `0` | `0–5+` | Minimum loss reduction required to make a further split. Higher values add conservatism. |
| `subsample` | `1.0` | `0.5–1.0` | Fraction of training instances sampled per tree. Lower values prevent overfitting. |
| `colsample_bytree` | `1.0` | `0.5–1.0` | Fraction of features sampled per tree. Prevents strong features from dominating all trees. |
| `reg_alpha` (`alpha`) | `0` | `0–10+` | L1 regularization on leaf weights. Promotes feature sparsity. |
| `reg_lambda` (`lambda`) | `1` | `0.1–10+` | L2 regularization on leaf weights. Smooths leaf weights to reduce variance. |
| `scale_pos_weight` | `1` | `neg_count / pos_count` | Controls balance of positive/negative weights for imbalanced classes. |

---

### Detailed Parameter Mechanics

#### 1. Tree Structure & Complexity

* **`max_depth`**: Defines how deep individual decision trees can grow. Deep trees capture complex high-order feature interactions but are prone to memorizing noise.
* **`min_child_weight`**: In classification, this corresponds to the minimum effective number of observations required in a leaf node. If a split results in a leaf with less hessian sum than this parameter, tree building stops partitioning.
* **`gamma`**: Acts as a split threshold penalty. XGBoost evaluates gain at each split; if `Gain < gamma`, the node is pruned.

#### 2. Stochastic Sampling

* **`subsample`**: Row subsampling ratio. Setting it to `0.8` means XGBoost randomly collects 80% of training instances before growing each tree.
* **`colsample_bytree`**: Column subsampling ratio. Samples a subset of features for constructing each individual tree.
* **`colsample_bylevel`**: Subsamples features at each depth level of a tree.
* **`colsample_bynode`**: Subsamples features at each split point/node.

#### 3. Regularization & Learning Rate

* **`learning_rate` (`eta`)**: Scales down the magnitude of feature weights after each boosting step. Shrinkage reduces the impact of individual trees and makes the overall model robust to overshooting loss minima.
* **`reg_alpha`**: L1 regularization term. Forces weak or uninformative leaf node weights to zero.
* **`reg_lambda`**: L2 regularization term. Penalizes extreme leaf weights, ensuring soft updates across splits.

#### 4. Execution & Engine Parameters

* **`tree_method`**:
* `'exact'`: Evaluates all split candidates (slow on large datasets).
* `'hist'`: Uses continuous feature histogram binning. Highly accelerated algorithm recommended for larger datasets and GPU processing (`device='cuda'`).


* **`n_jobs`**: Number of parallel CPU threads used during execution (`-1` leverages all CPU cores).

---

## Handling Imbalanced Data & Early Stopping

### Imbalanced Classification Setup

1. **`scale_pos_weight`**: Balance the gradient weights using:

$$\text{scale\_pos\_weight} = \frac{\text{Number of Negative Instances}}{\text{Number of Positive Instances}}$$


2. **`max_delta_step`**: Set to `1–10` when class imbalance is extreme. Constrains leaf updates to prevent extreme probability spikes.
3. **`eval_metric`**: Replace default `logloss` with ranking or precision metrics like `auc` or `aucpr`.

### Early Stopping Syntax

Halts tree creation when evaluation metrics stop improving over a given window:

```python
model = XGBClassifier(
    n_estimators=1000,
    learning_rate=0.03,
    early_stopping_rounds=15,
    eval_metric="logloss",
)

model.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)

```

---

## Sequential Tuning Workflow Strategy

1. **Establish Baseline**: Train with fixed `learning_rate=0.1` and default `n_estimators=100`.
2. **Tune Tree Structure**: Use `GridSearchCV` or `RandomizedSearchCV` on `max_depth` (`3–10`) and `min_child_weight` (`1–6`).
3. **Tune Split Penalty**: Search `gamma` (`0.0–0.5`) to prune unnecessary splits.
4. **Tune Subsampling**: Optimize `subsample` (`0.6–1.0`) and `colsample_bytree` (`0.6–1.0`) simultaneously.
5. **Tune Regularization**: Search `reg_alpha` and `reg_lambda` across log scales (`1e-3` to `10`).
6. **Lower Learning Rate**: Reduce `learning_rate` to `0.01–0.03` and increase `n_estimators` with early stopping.

In [4]:
import numpy as np
from scipy.stats import randint, uniform
from sklearn.datasets import load_breast_cancer
from sklearn.metrics import accuracy_score, classification_report
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV, train_test_split
from xgboost import XGBClassifier

# 1. Load Built-in Dataset & Split
data = load_breast_cancer()
X, y = data.data, data.target
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# ---------------------------------------------------------
# Part 1: Basic Model with Default Parameters
# ---------------------------------------------------------
model_default = XGBClassifier(random_state=42)
model_default.fit(X_train, y_train)

y_pred_default = model_default.predict(X_val)
print("--- Default Model ---")
print(f"Validation Accuracy: {accuracy_score(y_val, y_pred_default):.4f}")

--- Default Model ---
Validation Accuracy: 0.9561



# Part 2: Hyperparameter Tuning with GridSearchCV

In [5]:


param_grid = {
    'n_estimators': [50, 100, 200],
    'max_depth': [3, 5, 7],
    'learning_rate': [0.01, 0.1, 0.2],
    'subsample': [0.8, 1.0],
    'colsample_bytree': [0.8, 1.0],
}

grid_search = GridSearchCV(
    estimator=XGBClassifier(random_state=42, eval_metric='logloss'),
    param_grid=param_grid,
    cv=5,
    scoring='accuracy',
    n_jobs=-1,
    verbose=1,
)

grid_search.fit(X_train, y_train)
best_grid_model = grid_search.best_estimator_

print("\n--- GridSearchCV ---")
print(f"Best Parameters: {grid_search.best_params_}")
print(f"Best CV Accuracy: {grid_search.best_score_:.4f}")
print(
    f"Validation Accuracy: {accuracy_score(y_val, best_grid_model.predict(X_val)):.4f}"
)

Fitting 5 folds for each of 108 candidates, totalling 540 fits

--- GridSearchCV ---
Best Parameters: {'colsample_bytree': 1.0, 'learning_rate': 0.2, 'max_depth': 5, 'n_estimators': 100, 'subsample': 0.8}
Best CV Accuracy: 0.9824
Validation Accuracy: 0.9561


# Part 3: Hyperparameter Tuning with RandomizedSearchCV

In [6]:

param_distributions = {
    'n_estimators': randint(50, 300),
    'max_depth': randint(3, 10),
    'learning_rate': uniform(0.01, 0.2),
    'subsample': uniform(0.6, 0.4),
    'colsample_bytree': uniform(0.6, 0.4),
    'gamma': uniform(0, 0.5),
    'min_child_weight': randint(1, 6),
}

random_search = RandomizedSearchCV(
    estimator=XGBClassifier(random_state=42, eval_metric='logloss'),
    param_distributions=param_distributions,
    n_iter=25,
    cv=5,
    scoring='accuracy',
    n_jobs=-1,
    random_state=42,
    verbose=1,
)

random_search.fit(X_train, y_train)
best_random_model = random_search.best_estimator_

print("\n--- RandomizedSearchCV ---")
print(f"Best Parameters: {random_search.best_params_}")
print(f"Best CV Accuracy: {random_search.best_score_:.4f}")
print(
    f"Validation Accuracy: {accuracy_score(y_val, best_random_model.predict(X_val)):.4f}"
)

Fitting 5 folds for each of 25 candidates, totalling 125 fits

--- RandomizedSearchCV ---
Best Parameters: {'colsample_bytree': np.float64(0.9378135394712606), 'gamma': np.float64(0.37366005506869043), 'learning_rate': np.float64(0.11793842647781595), 'max_depth': 4, 'min_child_weight': 1, 'n_estimators': 237, 'subsample': np.float64(0.8170784332632994)}
Best CV Accuracy: 0.9802
Validation Accuracy: 0.9561
